In [1]:
import os
import sys
import glob
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
import matplotlib

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.lines as mlines
from matplotlib.colors import Normalize
from matplotlib.gridspec import GridSpec
from matplotlib.path import Path as MplPath
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import cartopy.crs as ccrs
import cartopy.feature as cfeature

from ladcp_tools import *

In [7]:
BASE = Path("/Users/emmagurcan/Documents/France/ENS/Speich/WHIRLS")
DIR_SADCP = BASE / "data/SADCP"
SADCP_SOURCE = "TOT"
SADCP_FILES = {
    "shallow": {"TOT": DIR_SADCP / "WHIRLS_NB150_TOT_osite.nc"},
    "deep":    {"TOT": DIR_SADCP / "WHIRLS_BB38_TOT_osite.nc"},
}

DIR_LADCP = BASE / "data/LADCP/traitement_avec_SADCP"
DIR_SSH = BASE / "data/SSH"
DIR_EDDIES = {"TOEDDIES_better_ADT": BASE / "data/TOEDDIES/TOEDDIES_better_ADT",
              "TOEDDIES": BASE / "data/TOEDDIES/TOEDDIES"}
FNAME_BATHY = BASE / "data/bathymetry/etopo2.nc"
DIR_TRACKS = BASE / "figures/eddy_tracks"
DIR_CODE = BASE / "analysis"          # where ladcp_tools.py lives
DIR_EDDIES_BETTER = DIR_EDDIES["TOEDDIES_better_ADT"]
DIR_EDDIES_STD = DIR_EDDIES["TOEDDIES"]
OUT_DIR = BASE / "figures/"
SADCP_TRACK_CSV = BASE / "figures/sadcp_vs_geostrophy/sadcp_vs_geostrophy_track.csv"

In [8]:
CRUISE_START = "2026-06-28"
CRUISE_END = "2026-07-27T23:59:59"

AVG_WINDOW = "1h"          # SADCP time averaging (reduces ping noise, not tides/inertial)
MIN_COVERAGE = 0.25        # keep a window if >= 25% of its expected ensembles are valid
                           # (expected count comes from each file's own sampling interval)
MIN_LAYER_FRAC = 0.5       # 0-100 m mean needs >= 50% of the bins that normally hold data
MAX_ADT_GAP = pd.Timedelta(hours=36)  # no geostrophy if the nearest ADT map is further away
Z_SURF = 30.0              # m — "near-surface" level
Z_LAYER = (50.0, 100.0)     # m — layer average
Z_SPLIT = 150.0            # m — use OS150 above, OS38 below for depth profiles
DEPTHS_SECTION = [30, 50, 75, 100, 150, 200, 300, 500] 
MIN_SPEED_ANGLE = 0.10     # m/s — angle differences below this speed are unreliable
ARROW_EVERY = "3h"         # arrow thinning on static maps
ARROW_SCALE = 2.0          # m/s per inch of arrow
ANIM_STEP = "2h"           # one animation frame every ANIM_STEP
ANIM_LEVEL = "layer"       # "layer" (0-100 m mean) or "surf" (30 m) for the ship arrows
ANIM_FPS = 6
ADD_LAND = os.environ.get("WHIRLS_NO_LAND") is None

# Optional named sections (e.g. eddy crossings) for extra Fig.7-like panels:
SECTIONS = {
    # "eddy_A_crossing": ("2026-07-10T00:00", "2026-07-12T18:00"),
}

G = 9.81
OMEGA = 7.2921e-5
R_EARTH = 6371e3



In [9]:
SADCP_SOURCE

'TOT'

In [ ]:
def load_sadcp_source(kind, source="TOT"):
    """Load the shallow/deep SADCP from TOT, MD2, or MD2 filled with TOT."""
    files = SADCP_FILES[kind]
    if source in ("TOT", "MD2"):
        return load_sadcp(files[source]) if files[source].exists() else None